# Investor Engagement Intelligence — Phase 1: Data Foundation

An AI-powered client retention and engagement analysis tool, built for IR and strategy teams.

**Business question:** Which client relationships are at risk, and what should the IR team do about it in the next 30 days?

This notebook pulls the [Telco Customer Churn dataset](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) (IBM sample data via Kaggle) and reframes it into IR-analog language.

> ⚠️ **Important:** this is a structural analog dataset, not real investor data — see the assumptions section below before drawing any conclusions from it.

## Step 1 — Pull the dataset

In [ ]:
!pip install kagglehub -q

import kagglehub
import pandas as pd
import numpy as np
import os

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
csv_path = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
raw = pd.read_csv(csv_path)

assert raw.shape == (7043, 21), f"Unexpected shape: {raw.shape}"
print("Loaded:", raw.shape)
raw.head()

## Step 2 — Clean

**Data quality note:** `TotalCharges` is stored as text and has 11 blank-string rows. All 11 have `tenure == 0` (brand-new clients, no completed billing cycle yet) — verified below with an assertion, not assumed. These are set to `0`, which is a real, defensible value here (no engagement cycle has occurred yet), not an imputed guess. None of these 11 have churned.

In [ ]:
blank_mask = raw["TotalCharges"].str.strip() == ""
print("Blank TotalCharges rows:", blank_mask.sum())
assert (raw.loc[blank_mask, "tenure"] == 0).all(), "Assumption broken — investigate"

cleaned = raw.copy()
cleaned["TotalCharges"] = cleaned["TotalCharges"].str.strip().replace("", "0").astype(float)
print("Clean complete.")

## Step 3 — Reframe into IR language

| Telco concept | IR concept |
|---|---|
| Subscriber tenure | Relationship length |
| Monthly/total billing | Engagement value / AUM proxy |
| Service add-ons subscribed | Engagement depth |
| Contract length | Commitment term |
| Customer churn | Redemption risk |

**Why this is a reasonable proxy:** all five are relationship-lifecycle signals — how long someone's been engaged, how much value they contribute, how many products/services they touch, how locked-in their commitment is, and whether they eventually leave. That lifecycle structure is domain-agnostic.

**Why this is NOT a validated model of investor behavior:** telco churn drivers (price, service quality) aren't proven to be the same as investor redemption drivers (fund performance, liquidity needs, IR relationship quality). This pipeline is built to demonstrate the analytical + AI-synthesis approach — transferable to real engagement data — not to claim a specific finding about *why* investors redeem.

Demographic fields (`Partner`, `Dependents`, `SeniorCitizen`) are kept as-is rather than force-translated into IR language — there's no honest client-relationship analog for them, and renaming would misrepresent the data.

In [ ]:
SERVICE_COLUMNS = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
]

def engagement_depth_score(row):
    return sum(1 for col in SERVICE_COLUMNS if row[col] == "Yes")

def aum_tier(monthly_value, q):
    if monthly_value <= q[0.25]:
        return "Tier 1 - Emerging"
    elif monthly_value <= q[0.5]:
        return "Tier 2 - Core"
    elif monthly_value <= q[0.75]:
        return "Tier 3 - Growth"
    else:
        return "Tier 4 - Strategic"

out = pd.DataFrame()
out["client_id"] = cleaned["customerID"]
out["relationship_length_months"] = cleaned["tenure"]
out["monthly_engagement_value"] = cleaned["MonthlyCharges"]
out["cumulative_relationship_value"] = cleaned["TotalCharges"]

q = cleaned["MonthlyCharges"].quantile([0.25, 0.5, 0.75]).to_dict()
out["aum_tier"] = cleaned["MonthlyCharges"].apply(lambda v: aum_tier(v, q))

out["commitment_term"] = cleaned["Contract"]
out["engagement_channel"] = cleaned["PaymentMethod"]
out["digital_engagement"] = (cleaned["PaperlessBilling"] == "Yes")
out["engagement_depth_score"] = cleaned.apply(engagement_depth_score, axis=1)
out["primary_product_line"] = cleaned["InternetService"]

out["household_partner"] = (cleaned["Partner"] == "Yes")
out["household_dependents"] = (cleaned["Dependents"] == "Yes")
out["senior_citizen"] = cleaned["SeniorCitizen"].astype(bool)

out["redemption_risk_flag"] = (cleaned["Churn"] == "Yes")

out.head()

## Step 4 — Validate

In [ ]:
assert out["client_id"].is_unique, "Duplicate client_id found"
assert out.isnull().sum().sum() == 0, "Unexpected nulls after reframing"
assert set(out["aum_tier"].unique()) == {
    "Tier 1 - Emerging", "Tier 2 - Core", "Tier 3 - Growth", "Tier 4 - Strategic"
}, "AUM tier bucketing produced unexpected categories"
assert out["engagement_depth_score"].between(0, len(SERVICE_COLUMNS)).all()

print("Validation passed.")
print(f"Rows: {len(out)}")
print(f"Redemption risk rate: {out['redemption_risk_flag'].mean():.1%}")
print(out["aum_tier"].value_counts())

## Step 5 — Save output

In [ ]:
out.to_csv("clients_ir.csv", index=False)
print("Saved clients_ir.csv")

# Uncomment to download in Colab:
# from google.colab import files
# files.download("clients_ir.csv")

## What downstream phases should NOT do

- Don't present `aum_tier` labels or `redemption_risk_flag` rates as reflective of any real fund's actual investor base.
- Don't let the AI intelligence layer (Phase 3) phrase output as if it discovered validated investor behavior — prompts should frame the data explicitly as illustrative/proxy, and outputs should be checked to make sure that framing survives into the generated report.

**Next: Phase 2 — analytical layer (redemption risk score by segment, engagement depth by relationship length, retention rate, early warning signals) + visualizations.**

---
# Phase 2 — Analytical Layer

Four metrics, each answering a piece of the business question: *which client relationships are at risk, and what should the IR team do in the next 30 days?*

1. Redemption risk by client segment (AUM tier × commitment term)
2. Engagement depth by relationship length
3. High-value client retention rate
4. Early warning signal (a composite flag for imminent risk)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## Metric 1 — Redemption risk by segment

Redemption risk broken out by AUM tier and commitment term. The interesting finding: **redemption risk is not lowest for the highest-value tier.** Tier 4 (Strategic) clients redeem at a higher rate than Tier 1 or Tier 2 — high value does not automatically mean high loyalty.

In [ ]:
segment = df.groupby(['aum_tier', 'commitment_term'], observed=True)['redemption_risk_flag'].mean().unstack()
segment = segment[['Month-to-month', 'One year', 'Two year']]  # consistent column order

print(segment.round(3))

ax = segment.plot(kind='bar', figsize=(9, 5))
ax.set_ylabel('Redemption risk rate')
ax.set_xlabel('AUM tier')
ax.set_title('Redemption risk by AUM tier and commitment term')
ax.legend(title='Commitment term')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Metric 2 — Engagement depth by relationship length

Engagement depth rises steadily with tenure, and redemption risk falls sharply — from ~47% in the first 12 months to ~7% after 5+ years. The first 12 months carry the bulk of the risk.

In [ ]:
bins = [-1, 12, 24, 36, 48, 60, 72]
labels = ['0-12', '13-24', '25-36', '37-48', '49-60', '61-72']
df['tenure_bucket'] = pd.cut(df['relationship_length_months'], bins=bins, labels=labels)

by_tenure = df.groupby('tenure_bucket', observed=True).agg(
    avg_engagement_depth=('engagement_depth_score', 'mean'),
    redemption_rate=('redemption_risk_flag', 'mean'),
)
print(by_tenure.round(3))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(by_tenure.index.astype(str), by_tenure['avg_engagement_depth'], color='#4C72B0', alpha=0.7, label='Avg engagement depth')
ax1.set_ylabel('Avg engagement depth (0-8)')
ax1.set_xlabel('Relationship length (months)')

ax2 = ax1.twinx()
ax2.plot(by_tenure.index.astype(str), by_tenure['redemption_rate'], color='#C44E52', marker='o', linewidth=2, label='Redemption rate')
ax2.set_ylabel('Redemption risk rate')

fig.suptitle('Engagement depth and redemption risk by relationship length')
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=2)
plt.tight_layout()
plt.show()

## Metric 3 — High-value client retention rate

Comparing Tier 4 (Strategic) retention to the overall book. This is the counterintuitive headline for leadership: **the highest-value segment retains worse than the client base as a whole.**

In [ ]:
overall_retention = 1 - df['redemption_risk_flag'].mean()
tier4_retention = 1 - df.loc[df['aum_tier'] == 'Tier 4 - Strategic', 'redemption_risk_flag'].mean()

print(f"Overall retention rate:        {overall_retention:.1%}")
print(f"Tier 4 (Strategic) retention:  {tier4_retention:.1%}")
print(f"Gap:                           {(overall_retention - tier4_retention):.1%} pts worse for highest-value clients")

fig, ax = plt.subplots(figsize=(5, 5))
ax.bar(['Overall book', 'Tier 4 - Strategic'], [overall_retention, tier4_retention], color=['#55A868', '#C44E52'])
ax.set_ylabel('Retention rate')
ax.set_ylim(0, 1)
ax.set_title('Retention: overall book vs. highest-value tier')
for i, v in enumerate([overall_retention, tier4_retention]):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Metric 4 — Early warning signal

A composite flag, not a single variable: clients who are simultaneously
- **low engagement** (engagement depth ≤ 2, at/below the 25th percentile)
- **short relationship** (≤ 12 months)
- **high value** (monthly engagement value ≥ median)

This is a deliberately narrow, high-precision flag — the goal is a short, actionable list for the IR team, not a broad net.

In [ ]:
median_value = df['monthly_engagement_value'].median()

warning_mask = (
    (df['engagement_depth_score'] <= 2) &
    (df['relationship_length_months'] <= 12) &
    (df['monthly_engagement_value'] >= median_value)
)

df['early_warning_flag'] = warning_mask

n_flagged = warning_mask.sum()
pct_flagged = warning_mask.mean()
warning_redemption = df.loc[warning_mask, 'redemption_risk_flag'].mean()
baseline_redemption = df.loc[~warning_mask, 'redemption_risk_flag'].mean()

print(f"Clients flagged:              {n_flagged} ({pct_flagged:.1%} of client base)")
print(f"Redemption rate — flagged:    {warning_redemption:.1%}")
print(f"Redemption rate — baseline:   {baseline_redemption:.1%}")
print(f"Lift:                         {warning_redemption / baseline_redemption:.1f}x baseline")

fig, ax = plt.subplots(figsize=(5, 5))
ax.bar(['Baseline clients', 'Early warning flag'], [baseline_redemption, warning_redemption], color=['#8C8C8C', '#C44E52'])
ax.set_ylabel('Redemption risk rate')
ax.set_ylim(0, 1)
ax.set_title(f'Early warning segment vs. baseline\n({n_flagged} clients, {pct_flagged:.1%} of book)')
for i, v in enumerate([baseline_redemption, warning_redemption]):
    ax.text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 2 summary — what this tells the IR team

- **Redemption risk concentrates in the first 12 months**, and drops off sharply after — retention efforts have the highest ROI early in a relationship.
- **High AUM does not equal loyalty.** Tier 4 (Strategic) clients redeem at a higher rate than Tier 1 or Tier 2, and their overall retention rate is worse than the book average — this segment needs more attention, not less.
- **Commitment term is a strong protective factor** — month-to-month clients redeem at ~15x the rate of two-year clients — but it's also likely a symptom, not just a cause, of a weaker relationship.
- **A narrow, composite early-warning flag** (low engagement + short tenure + high value) isolates ~5% of the book with a redemption rate nearly 3x baseline — small enough for the IR team to act on individually within 30 days.

*Reminder: these patterns are illustrative of the analytical approach, derived from a proxy dataset — see the Phase 1 assumptions section for what this can and can't claim about real investor behavior.*

**Next: Phase 3 — AI intelligence layer (structured summary → Claude API → plain-English strategic recommendations, with a full audit trail).**